# Risk scoring and explainability

Up to this point we've evaluated models in isolation. A risk scoring framework is the layer that turns model output into operational decisions: scores get bucketed into bands, bands get mapped to actions (allow / monitor / review / block), and individual high-risk transactions get explanations that an analyst can act on.

This notebook combines the supervised XGBoost model from `03_model_baseline.ipynb` with the Isolation Forest from `04_anomaly_detection.ipynb` into a single risk score, defines bands, demonstrates per-transaction explanations with SHAP, and discusses what a real fintech deployment would still need beyond what is implemented here.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "src" / "models"
FIGURES_DIR = Path("..") / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
test = pd.read_csv(PROCESSED_DIR / "test.csv")
X_test = test.drop(columns=["isFraud"])
y_test = test["isFraud"]

xgb_model = joblib.load(MODELS_DIR / "xgb_baseline.pkl")
iso_model = joblib.load(MODELS_DIR / "isolation_forest.pkl")

print(f"Test set: {X_test.shape}, fraud rate {y_test.mean():.4%}")
